In [ ]:
%config InlineBackend.figure_formats = {"retina", "png"}
%matplotlib inline

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import tempfile

from IPython.display import HTML, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import h5py

import tdgl
from tdgl.geometry import box, circle

In [ ]:
tempdir = tempfile.TemporaryDirectory(dir=os.getcwd())

# Definición de parámetros del material y de la geometría de peine asimétrico

La geometría sigue la Ec. (1) del paper:
$$y(x) = -h_d \sum_{i=1}^{N} \exp\left(-\frac{(x - x_i)^2}{2\delta^2}\right)$$

donde:
- $h_d$ = profundidad máxima de los dientes
- $x_i$ = posición central del i-ésimo diente
- $\delta = 0.20 \cdot \Delta x$ = parámetro de suavidad geométrica
- $N$ = número de dientes (controla la densidad del peine)

In [ ]:
length_units = "um"

# ─────────────────────────────────────────────
# PARÁMETROS DEL MATERIAL (sin cambios)
# ─────────────────────────────────────────────
xi            = 0.5   # Longitud de coherencia (µm)
london_lambda = 2     # Longitud de penetración de London (µm)
d             = 0.1   # Espesor de la lámina (µm)
gamma         = 1     # Parámetro de amortiguamiento

layer = tdgl.Layer(
    coherence_length=xi,
    london_lambda=london_lambda,
    thickness=d,
    gamma=gamma
)

# ─────────────────────────────────────────────
# PARÁMETROS GEOMÉTRICOS GLOBALES DEL DISPOSITIVO
# Según el paper: Lx = 20 µm, Ly = 5 µm, Lz = 0.1 µm
# ─────────────────────────────────────────────
total_length = 20.0   # Lx: longitud total del dispositivo (µm)
total_width  = 5.0    # Ly: altura del bloque rectangular base (µm)

# ─────────────────────────────────────────────
# PARÁMETROS DE LOS DIENTES GAUSSIANOS (Ec. 1 del paper)
# ─────────────────────────────────────────────
N  = 8      # Número de dientes (probar: 4, 8, 12, 16)
hd = 3.5    # Profundidad máxima de cada diente (µm)
            # Nota: hd < total_width para que el dispositivo no quede cerrado

# Separación entre dientes: se distribuyen uniformemente a lo largo de Lx
delta_x = total_length / N          # Separación entre centros de dientes (µm)
delta   = 0.20 * delta_x            # Parámetro de suavidad geométrica (δ = 0.20·Δx)

# Posiciones centrales de cada diente xi, distribuidas uniformemente
# Se centra el primer y último diente a media separación del borde
x_teeth = np.linspace(
    -total_length / 2 + delta_x / 2,
     total_length / 2 - delta_x / 2,
    N
)

print(f"Número de dientes    N  = {N}")
print(f"Separación entre dientes Δx = {delta_x:.3f} µm")
print(f"Parámetro de suavidad    δ  = {delta:.3f} µm")
print(f"Profundidad de dientes   hd = {hd} µm")
print(f"Posiciones xi = {np.round(x_teeth, 2)}")

# ─────────────────────────────────────────────
# FUNCIÓN DEL PERFIL DEL BORDE INFERIOR (Ec. 1)
# y(x) = -hd * Σ exp(-(x - xi)^2 / (2δ^2))
# ─────────────────────────────────────────────
def lower_edge_profile(x, hd, x_teeth, delta):
    """Perfil del borde inferior con dientes gaussianos (Ec. 1 del paper)."""
    y = np.zeros_like(x, dtype=float)
    for xi_i in x_teeth:
        y += -hd * np.exp(-((x - xi_i) ** 2) / (2 * delta ** 2))
    return y

# ─────────────────────────────────────────────
# CONSTRUCCIÓN DEL POLÍGONO DE PEINE ASIMÉTRICO
# El polígono recorre: borde superior (recto) → lado derecho → 
#   borde inferior con dientes → lado izquierdo → cierre
# ─────────────────────────────────────────────
n_points = 1000  # Resolución del borde inferior (más puntos = dientes más suaves)

x_bottom = np.linspace(-total_length / 2, total_length / 2, n_points)
y_bottom = lower_edge_profile(x_bottom, hd, x_teeth, delta)

# El borde inferior tiene coordenadas y en [-total_width/2 + y_diente]
# El borde superior está en y = +total_width/2 (recto, sin dientes)
y_top    =  total_width / 2
y_base   = -total_width / 2   # Línea base del borde inferior sin dientes

# El perfil gaussiano desplaza el borde inferior hacia abajo desde y_base
y_bottom_coords = y_base + y_bottom  # y_bottom es negativo → borde baja

# Ensamblar el contorno completo del polígono (sentido antihorario)
#   1. Borde superior: de izquierda a derecha
#   2. Lado derecho: baja de y_top a y_base
#   3. Borde inferior con dientes: de derecha a izquierda
#   4. Lado izquierdo: sube de y_base a y_top
x_top_edge    = np.array([-total_length/2, total_length/2])
y_top_edge    = np.array([y_top, y_top])

x_right_edge  = np.array([total_length/2, total_length/2])
y_right_edge  = np.array([y_top, y_base])

# Borde inferior (de derecha a izquierda, para cerrar el polígono)
x_bot_rev     = x_bottom[::-1]
y_bot_rev     = y_bottom_coords[::-1]

x_left_edge   = np.array([-total_length/2, -total_length/2])
y_left_edge   = np.array([y_base, y_top])

# Concatenar todos los segmentos
x_polygon = np.concatenate([x_top_edge, x_right_edge, x_bot_rev, x_left_edge])
y_polygon = np.concatenate([y_top_edge, y_right_edge, y_bot_rev, y_left_edge])

polygon_points = np.column_stack([x_polygon, y_polygon])

# ─────────────────────────────────────────────
# FILM: Polígono de peine asimétrico
# ─────────────────────────────────────────────
film = (
    tdgl.Polygon("film", points=polygon_points)
    .resample(1200)   # Mayor resolución para capturar bien los dientes
    .buffer(0)
)

# ─────────────────────────────────────────────
# TERMINALES: Source (izquierda) y Drain (derecha)
# Longitud del contacto = 2.0 µm según el paper
# ─────────────────────────────────────────────
contact_length = 2.0   # Longitud de los contactos metálicos (µm)
contact_width  = 0.1   # Ancho del contacto en x (µm, muy delgado)

source = (
    tdgl.Polygon("source", points=box(contact_width, contact_length))
    .translate(dx=-total_length / 2)
)

drain = source.scale(xfact=-1).set_name("drain")

# ─────────────────────────────────────────────
# PUNTOS DE MEDICIÓN DE VOLTAJE
# ─────────────────────────────────────────────
probe_points = [
    (-total_length / 2.5, 0),
     (total_length / 2.5, 0)
]

# ─────────────────────────────────────────────
# DEFINICIÓN DEL DISPOSITIVO
# ─────────────────────────────────────────────
device = tdgl.Device(
    "peine_asimetrico",
    layer=layer,
    film=film,
    terminals=[source, drain],
    probe_points=probe_points,
    length_units=length_units,
)

print("\nDispositivo creado correctamente.")

In [ ]:
# Visualizar el perfil del borde inferior antes de crear el dispositivo
fig, ax = plt.subplots(figsize=(10, 3))
x_plot = np.linspace(-total_length / 2, total_length / 2, 2000)
y_plot = y_base + lower_edge_profile(x_plot, hd, x_teeth, delta)
ax.plot(x_plot, y_plot, 'b-', label='Borde inferior (dientes)')
ax.axhline(y_top, color='r', linestyle='--', label='Borde superior')
ax.fill_between(x_plot, y_plot, y_top, alpha=0.15, color='gray')
for xi_i in x_teeth:
    ax.axvline(xi_i, color='g', linestyle=':', alpha=0.5, linewidth=0.8)
ax.set_xlabel('x (µm)')
ax.set_ylabel('y (µm)')
ax.set_title(f'Geometría de peine asimétrico — N={N} dientes, hd={hd} µm, δ={delta:.2f} µm')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Dibujar el dispositivo con tdgl
fig, ax = device.draw()

In [ ]:
# ─────────────────────────────────────────────
# OPCIONES DEL SOLVER (sin cambios)
# ─────────────────────────────────────────────
options = tdgl.SolverOptions(
    skip_time=100,
    solve_time=200,
    output_file=os.path.join(tempdir.name, "campo_corriente_peine.h5"),
    field_units="mT",
    current_units="uA",
    save_every=100,
)

In [ ]:
# Generar la malla del dispositivo
device.make_mesh(max_edge_length=xi / 2, smooth=100)
fig, ax = device.plot(mesh=True, legend=True)

# Simulación de las corrientes en el superconductor de peine

- I = 12 µA
- H = 0

In [ ]:
default_terminal_currents = {
    'source':  12,
    'drain':  -12
}

H_zero_solution = tdgl.solve(
    device=device,
    options=options,
    terminal_currents=default_terminal_currents
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
_ = H_zero_solution.plot_currents(ax=axes[0], streamplot=False)
_ = H_zero_solution.plot_currents(ax=axes[1])

# Sección transversal vertical en el centro del dispositivo
y_cross = np.linspace(-total_width / 2, total_width / 2, 401)
x_cross = np.zeros_like(y_cross)
cross_section = np.array([x_cross, y_cross]).T

for ax in axes:
    _ = ax.plot(x_cross, y_cross, "C1--")

current = H_zero_solution.current_through_path(cross_section)
print(f"Corriente medida: {current:.3f~P}")

In [ ]:
# ─────────────────────────────────────────────
# BARRIDO DE CORRIENTE: Curva V vs I
# ─────────────────────────────────────────────
def get_mean_voltage(device, options, current):
    terminal_currents = {
        'source':  current,
        'drain':  -current
    }
    solution = tdgl.solve(
        device=device,
        options=options,
        terminal_currents=terminal_currents
    )
    voltage = solution.dynamics.mean_voltage()
    return voltage

currents = np.arange(0.0, 50, 0.1)
voltages = []
intensities = []

voltage_current_csv_filename = "V_vs_I_peine.csv"
csv_path = os.path.join("raw_data", voltage_current_csv_filename)

if os.path.exists(csv_path):
    os.remove(csv_path)

for current in currents:
    voltage = get_mean_voltage(device=device, options=options, current=current)
    voltages.append(voltage)
    intensities.append(current)

    csv_df = pd.DataFrame({'current_µA': [current], 'voltage_V': [voltage]})
    csv_df.to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)

In [ ]:
df = pd.read_csv(csv_path)
arr = df.to_numpy()

voltages    = [a[1] for a in arr]
intensities = [a[0] for a in arr]

plt.figure(figsize=(6, 4))
plt.plot(intensities, voltages, linestyle='-', color='blue')
plt.xlabel('Corriente (µA)')
plt.ylabel('Voltaje promedio (V)')
plt.title(f'Curva V-I — Peine asimétrico N={N}')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(intensities[:300], voltages[:300], linestyle='-', color='blue')
plt.xlabel('Corriente (µA)')
plt.ylabel('Voltaje promedio (V)')
plt.title(f'Curva V-I (primeros 300 puntos) — N={N}')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(intensities[:240], voltages[:240], color='blue')
plt.xlabel('Corriente (µA)')
plt.ylabel('Voltaje promedio (V)')
plt.title(f'Curva V-I (primeros 240 puntos) — N={N}')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# DETECCIÓN DE LA CORRIENTE CRÍTICA
# Criterio: mayor salto (gap) consecutivo en voltaje
# ─────────────────────────────────────────────
critical_current = 0
biggest_gap = 0

for i in range(len(voltages) - 1):
    tmp_gap = voltages[i + 1] - voltages[i]
    if tmp_gap > biggest_gap:
        biggest_gap = tmp_gap
        critical_current = i

plt.figure(figsize=(6, 4))
plt.plot(intensities[220:280], voltages[220:280], linestyle='-', color='blue')
plt.axvline(
    currents[critical_current],
    label=f'Corriente crítica: {currents[critical_current]:.2f} µA'
)
plt.xlabel('Corriente (µA)')
plt.ylabel('Voltaje promedio (V)')
plt.title(f'Corriente crítica — Peine N={N}')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Corriente crítica estimada: {currents[critical_current]:.2f} µA")